# Notebook 2 — Preference Alignment with DPO

After SFT teaches the task, **DPO** refines *judgment between* good and bad responses, learning directly
from `(prompt, chosen, rejected)` triples — no reward model, no RL loop. Runs on a free T4 with LoRA.

Set Runtime -> T4 GPU first.


## 1. Install


In [ ]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets

## 2. Load your SFT model (or a base) in 4-bit, add LoRA
Standard recipe: SFT first, then DPO on top. Point `model_name` at your SFT output if you have one.


In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model, r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj"],
    lora_alpha = 16, use_gradient_checkpointing = "unsloth",
)

## 3. Load preference data
Each line of `prefs.jsonl`: `{"prompt": "...", "chosen": "...", "rejected": "..."}`.
For the Twin: `chosen` = your-voice answer, `rejected` = a generic/off-voice answer.


In [ ]:
from datasets import load_dataset
prefs = load_dataset("json", data_files="prefs.jsonl", split="train")
print(prefs[0])
print("pairs:", len(prefs))

## 4. Train with DPO
Watch `rewards/accuracies` rise toward 1.0 (the model increasingly prefers `chosen`) and
the loss fall. `beta` controls how hard it pushes toward preferences (0.1 is a common default).


In [ ]:
from trl import DPOTrainer, DPOConfig

trainer = DPOTrainer(
    model = model,
    ref_model = None,        # LoRA uses the frozen base as the implicit reference
    tokenizer = tokenizer,
    train_dataset = prefs,
    args = DPOConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        num_train_epochs = 1,
        learning_rate = 5e-5,
        beta = 0.1,
        logging_steps = 1,
        output_dir = "dpo_outputs",
    ),
)
trainer.train()
model.save_pretrained("dpo_adapter")

---
**You aligned a model to preferences with DPO.** Now evaluate it in Notebook 3.
